# Data Inventory and Validation

**PartnerLens Copilot — Final Submission Notebook**

This notebook uses repository-relative paths and does not require Google Drive. Run it from a cloned copy of the `partnerlens-copilot` repository after installing `requirements.txt`.

## Environment setup

From a terminal opened at the repository root, install dependencies once:

```bash
python -m pip install -r requirements.txt
```

The next cell locates the repository automatically when Jupyter is started from the repository or its `notebooks` folder.

In [1]:
from pathlib import Path
import importlib.util
import os
import sys


def locate_repository_root() -> Path:
    """Locate the PartnerLens repository without relying on Google Drive."""
    current = Path.cwd().resolve()
    candidates = [
        current,
        *current.parents,
        Path.home() / "partnerlens-copilot",
        Path("/content/partnerlens-copilot"),
    ]

    visited = set()
    for candidate in candidates:
        try:
            candidate = candidate.expanduser().resolve()
        except OSError:
            continue
        if candidate in visited:
            continue
        visited.add(candidate)
        if (
            (candidate / "README.md").is_file()
            and (candidate / "src").is_dir()
            and (candidate / "data").is_dir()
        ):
            return candidate

    raise FileNotFoundError(
        "PartnerLens repository root was not found. Start Jupyter from inside "
        "the partnerlens-copilot repository, or clone the repository to "
        "~/partnerlens-copilot or /content/partnerlens-copilot."
    )


REPO_ROOT = locate_repository_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

required_modules = {
    "pandas": "pandas",
    "numpy": "numpy",
    "sqlparse": "sqlparse",
    "openpyxl": "openpyxl",
}
missing_packages = [
    package_name
    for module_name, package_name in required_modules.items()
    if importlib.util.find_spec(module_name) is None
]
if missing_packages:
    raise ModuleNotFoundError(
        "Missing required packages: "
        + ", ".join(missing_packages)
        + ". From the repository root run: "
        + f"{sys.executable} -m pip install -r requirements.txt"
    )

RAW_DIR = REPO_ROOT / "data" / "raw"
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
CONFIG_DIR = REPO_ROOT / "configs"
NOTEBOOK_OUTPUT_DIR = REPO_ROOT / "artifacts" / "notebook_outputs"
EVALUATION_OUTPUT_DIR = REPO_ROOT / "artifacts" / "evaluation"
NOTEBOOK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EVALUATION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {REPO_ROOT}")
print(f"Python executable: {sys.executable}")
print("Notebook environment is ready.")

Repository root: C:\Users\nafee\partnerlens-copilot
Python executable: C:\Users\nafee\AppData\Local\Programs\Python\Python312\python.exe
Notebook environment is ready.


## 1. Canonical dataset inventory

The canonical SQLite table names used across the notebooks and `src` modules are:

- `partners`
- `partner_pricing`
- `monthly_partner_metrics`
- `partner_current_preview`

The CSV filenames may differ from their SQLite table names, but this mapping is used consistently throughout the project.

In [2]:
import json
import pandas as pd
import numpy as np
from IPython.display import display

raw_files = {
    "partners": RAW_DIR / "partners.csv",
    "partner_pricing": RAW_DIR / "partner_pricing.csv",
    "pricing_plans": RAW_DIR / "pricing_plans.csv",
    "pricing_exceptions": RAW_DIR / "pricing_exceptions.csv",
    "monthly_partner_metrics": RAW_DIR / "monthly_partner_metrics.csv",
    "partner_segments": RAW_DIR / "partner_segments.csv",
}

processed_files = {
    "partners": PROCESSED_DIR / "partner_master_clean.csv",
    "partner_pricing": PROCESSED_DIR / "partner_pricing_clean.csv",
    "monthly_partner_metrics": PROCESSED_DIR / "partner_metrics_clean.csv",
    "partner_current_preview": PROCESSED_DIR / "partner_current_preview_1000.csv",
}

all_expected_files = {**{f"raw::{k}": v for k, v in raw_files.items()},
                      **{f"processed::{k}": v for k, v in processed_files.items()}}

missing_files = [str(path.relative_to(REPO_ROOT)) for path in all_expected_files.values() if not path.is_file()]
if missing_files:
    raise FileNotFoundError(
        "Required repository data files are missing:\n- " + "\n- ".join(missing_files)
    )

print(f"Validated {len(all_expected_files)} expected repository data files.")

Validated 10 expected repository data files.


In [3]:
def read_csv_preserving_ids(path: Path) -> pd.DataFrame:
    header = pd.read_csv(path, nrows=0)
    identifier_types = {
        column: "string"
        for column in header.columns
        if str(column).strip().endswith("_id")
    }
    dataframe = pd.read_csv(path, dtype=identifier_types, low_memory=False)
    dataframe.columns = [str(column).strip() for column in dataframe.columns]
    return dataframe


raw_data = {name: read_csv_preserving_ids(path) for name, path in raw_files.items()}
processed_data = {name: read_csv_preserving_ids(path) for name, path in processed_files.items()}

inventory_rows = []
for layer, datasets, paths in (
    ("raw", raw_data, raw_files),
    ("processed", processed_data, processed_files),
):
    for table_name, dataframe in datasets.items():
        inventory_rows.append(
            {
                "layer": layer,
                "canonical_table_name": table_name,
                "file_name": paths[table_name].name,
                "row_count": len(dataframe),
                "column_count": len(dataframe.columns),
                "duplicate_column_names": int(dataframe.columns.duplicated().sum()),
                "file_size_mb": round(paths[table_name].stat().st_size / (1024 ** 2), 3),
            }
        )

inventory_df = pd.DataFrame(inventory_rows).sort_values(
    ["layer", "canonical_table_name"]
).reset_index(drop=True)
display(inventory_df)

,layer,canonical_table_name,file_name,row_count,column_count,duplicate_column_names,file_size_mb
0,processed,monthly_partner_metrics,partner_metrics_clean.csv,120000,20,0,17.233
1,processed,partner_current_preview,partner_current_preview_1000.csv,1000,47,0,0.456
2,processed,partner_pricing,partner_pricing_clean.csv,5000,16,0,0.577
3,processed,partners,partner_master_clean.csv,5000,20,0,0.805
4,raw,monthly_partner_metrics,monthly_partner_metrics.csv,120000,14,0,11.712
5,raw,partner_pricing,partner_pricing.csv,5000,14,0,0.504
6,raw,partner_segments,partner_segments.csv,5000,4,0,0.687
7,raw,partners,partners.csv,5000,18,0,0.725
8,raw,pricing_exceptions,pricing_exceptions.csv,878,9,0,0.100
9,raw,pricing_plans,pricing_plans.csv,9,10,0,0.002


## 2. Schema and key validation

This section checks required columns, critical identifiers, duplicate business keys, numeric parsing, synthetic-data flags, and cross-table `partner_id` integrity.

In [4]:
with (CONFIG_DIR / "schema_metadata.json").open("r", encoding="utf-8") as schema_file:
    schema_metadata = json.load(schema_file)

canonical_required_columns = {
    "partners": set(schema_metadata["tables"]["partners"]),
    "partner_pricing": set(schema_metadata["tables"]["partner_pricing"]),
    "monthly_partner_metrics": set(schema_metadata["tables"]["monthly_partner_metrics"]),
    "partner_current_preview": {
        "partner_id",
        "partner_name",
        "industry_vertical",
        "state",
        "txn_growth_pct",
        "payment_volume_usd",
        "txn_count",
        "net_revenue_usd",
    },
}

business_keys = {
    "partners": ["partner_id"],
    "partner_pricing": ["assignment_id"],
    "monthly_partner_metrics": ["partner_id", "month"],
    "partner_current_preview": ["partner_id"],
}

critical_columns = {
    "partners": ["partner_id", "partner_name", "synthetic_record_flag"],
    "partner_pricing": ["assignment_id", "partner_id", "pricing_plan_id"],
    "monthly_partner_metrics": ["partner_id", "month"],
    "partner_current_preview": ["partner_id", "partner_name"],
}

numeric_columns = {
    "partner_pricing": [
        "negotiated_bps",
        "negotiated_per_txn_fee_usd",
        "monthly_minimum_fee_usd",
    ],
    "monthly_partner_metrics": [
        "txn_count",
        "payment_volume_usd",
        "txn_growth_pct",
        "net_revenue_usd",
    ],
    "partner_current_preview": [
        "txn_count",
        "payment_volume_usd",
        "txn_growth_pct",
        "net_revenue_usd",
    ],
}

validation_checks = []


def add_check(dataset: str, check_name: str, passed: bool, details: str) -> None:
    validation_checks.append(
        {
            "dataset": dataset,
            "check_name": check_name,
            "status": "PASS" if passed else "FAIL",
            "details": details,
        }
    )


for dataset_name, dataframe in processed_data.items():
    missing_columns = sorted(canonical_required_columns[dataset_name] - set(dataframe.columns))
    add_check(
        dataset_name,
        "required_columns",
        not missing_columns,
        "All required columns present."
        if not missing_columns
        else "Missing: " + ", ".join(missing_columns),
    )

    available_keys = [column for column in business_keys[dataset_name] if column in dataframe.columns]
    duplicate_count = (
        int(dataframe.duplicated(subset=available_keys, keep=False).sum())
        if len(available_keys) == len(business_keys[dataset_name])
        else -1
    )
    add_check(
        dataset_name,
        "duplicate_business_key",
        duplicate_count == 0,
        f"Duplicate-key rows: {duplicate_count}",
    )

    for column in critical_columns[dataset_name]:
        if column not in dataframe.columns:
            continue
        missing_mask = dataframe[column].isna() | dataframe[column].astype("string").str.strip().eq("")
        missing_count = int(missing_mask.fillna(True).sum())
        add_check(
            dataset_name,
            f"critical_field::{column}",
            missing_count == 0,
            f"Missing or blank values: {missing_count}",
        )

    for column in numeric_columns.get(dataset_name, []):
        if column not in dataframe.columns:
            continue
        populated = dataframe[column].notna() & dataframe[column].astype("string").str.strip().ne("")
        converted = pd.to_numeric(dataframe[column], errors="coerce")
        invalid_count = int((populated & converted.isna()).sum())
        add_check(
            dataset_name,
            f"numeric_field::{column}",
            invalid_count == 0,
            f"Nonnumeric populated values: {invalid_count}",
        )

partners_df = processed_data["partners"]
master_partner_ids = set(partners_df["partner_id"].dropna().astype(str).str.strip())
for child_name in ("partner_pricing", "monthly_partner_metrics", "partner_current_preview"):
    child_ids = set(processed_data[child_name]["partner_id"].dropna().astype(str).str.strip())
    orphan_ids = sorted(child_ids - master_partner_ids)
    add_check(
        child_name,
        "partner_id_referential_integrity",
        not orphan_ids,
        "All partner IDs exist in partners."
        if not orphan_ids
        else f"Orphan IDs found: {orphan_ids[:5]}",
    )

if "synthetic_record_flag" in partners_df.columns:
    accepted_flags = {"1", "true", "yes", "y", "synthetic"}
    normalized_flags = partners_df["synthetic_record_flag"].astype("string").str.strip().str.lower()
    invalid_flags = int((~normalized_flags.isin(accepted_flags)).sum())
    add_check(
        "partners",
        "synthetic_record_flag",
        invalid_flags == 0,
        f"Unexpected flag values: {invalid_flags}",
    )

validation_df = pd.DataFrame(validation_checks)
display(validation_df)

,dataset,check_name,status,details
0,partners,required_columns,PASS,All required columns present.
1,partners,duplicate_business_key,PASS,Duplicate-key rows: 0
2,partners,critical_field::partner_id,PASS,Missing or blank values: 0
3,partners,critical_field::partner_name,PASS,Missing or blank values: 0
4,partners,critical_field::synthetic_record_flag,PASS,Missing or blank values: 0
5,partner_pricing,required_columns,PASS,All required columns present.
6,partner_pricing,duplicate_business_key,PASS,Duplicate-key rows: 0
7,partner_pricing,critical_field::assignment_id,PASS,Missing or blank values: 0
8,partner_pricing,critical_field::partner_id,PASS,Missing or blank values: 0
9,partner_pricing,critical_field::pricing_plan_id,PASS,Missing or blank values: 0


## 3. Validation metrics and repository evidence

The notebook writes compact, machine-readable evidence under `artifacts/notebook_outputs/`. These outputs can be committed to GitHub so evaluators can see the computed metrics without rerunning the notebook.

In [5]:
validation_metrics = pd.DataFrame(
    [
        {
            "metric": "total_checks",
            "value": len(validation_df),
        },
        {
            "metric": "passed_checks",
            "value": int((validation_df["status"] == "PASS").sum()),
        },
        {
            "metric": "failed_checks",
            "value": int((validation_df["status"] == "FAIL").sum()),
        },
        {
            "metric": "validation_pass_rate_pct",
            "value": round(100 * (validation_df["status"] == "PASS").mean(), 2),
        },
        {
            "metric": "processed_rows_validated",
            "value": int(sum(len(df) for df in processed_data.values())),
        },
    ]
)

inventory_path = NOTEBOOK_OUTPUT_DIR / "data_inventory.csv"
checks_path = NOTEBOOK_OUTPUT_DIR / "data_validation_checks.csv"
metrics_path = NOTEBOOK_OUTPUT_DIR / "data_validation_metrics.csv"
workbook_path = NOTEBOOK_OUTPUT_DIR / "partnerlens_validation_review.xlsx"

inventory_df.to_csv(inventory_path, index=False)
validation_df.to_csv(checks_path, index=False)
validation_metrics.to_csv(metrics_path, index=False)

with pd.ExcelWriter(workbook_path, engine="openpyxl") as writer:
    inventory_df.to_excel(writer, sheet_name="inventory", index=False)
    validation_df.to_excel(writer, sheet_name="validation_checks", index=False)
    validation_metrics.to_excel(writer, sheet_name="metrics", index=False)
    processed_data["partner_current_preview"].head(100).to_excel(
        writer, sheet_name="preview_sample", index=False
    )

print("Saved notebook evidence:")
for output_path in (inventory_path, checks_path, metrics_path, workbook_path):
    print(f"- {output_path.relative_to(REPO_ROOT)}")

display(validation_metrics)

if (validation_df["status"] == "FAIL").any():
    raise AssertionError("One or more required data-validation checks failed.")

print("All required data-validation checks passed.")

Saved notebook evidence:
- artifacts\notebook_outputs\data_inventory.csv
- artifacts\notebook_outputs\data_validation_checks.csv
- artifacts\notebook_outputs\data_validation_metrics.csv
- artifacts\notebook_outputs\partnerlens_validation_review.xlsx


,metric,value
0,total_checks,33.0
1,passed_checks,33.0
2,failed_checks,0.0
3,validation_pass_rate_pct,100.0
4,processed_rows_validated,131000.0


All required data-validation checks passed.


## Final interpretation

A successful run demonstrates that the repository contains the expected synthetic datasets, the processed schemas align with the canonical project schema, critical keys are populated and unique, and child tables reference valid partner records. The generated CSV and Excel evidence should be committed for the final submission.